## Model Verification and Image Preprocessing

In [ ]:
# Install OpenCV if not already installed (usually pre-installed in Colab)
!pip install opencv-python

In [ ]:
import torch
import os
import pandas as pd

# Define the path to the poisoned model
# Changed for Kaggle environment
poisoned_model_path = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth'

# Load the poisoned model using PyTorch
try:
    # Ensure PyTorch is available and the model file exists
    if not os.path.exists(poisoned_model_path):
        raise FileNotFoundError(f"Model file not found at: {poisoned_model_path}")

    # Load the model state dictionary
    # Map location ensures it loads correctly even if trained on GPU but loaded on CPU
    poisoned_model_state_dict = torch.load(poisoned_model_path, map_location=torch.device('cpu'))
    print("Poisoned model state dictionary loaded successfully!")

    # For a general PyTorch model, we can print the keys of the state_dict
    # to understand its layers/components.
    print("Keys in the model's state dictionary:")
    for key in poisoned_model_state_dict.keys():
        print(key)

    print("\nNote: To fully verify the architecture and parameters, a corresponding PyTorch model class definition would be needed to reconstruct the model and then load this state_dict.")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please verify the model path.")
except Exception as e:
    print(f"Error loading PyTorch model: {e}")
    print("Please ensure the model path is correct and it's a valid PyTorch model file.")

### Debugging Model Loading Path

The error indicates the `saved_model.pb` file is missing. Let's inspect the contents of the `poisoned_model` directory to find the correct path.

In [ ]:
print(f"Contents of {poisoned_model_path}:")
!ls -F "{poisoned_model_path}"

# If there are subdirectories (e.g., '1'), we might need to look inside them.
# For example:
# !ls -F "{poisoned_model_path}/1/"

### Debugging Model Loading Path

The error indicates the `saved_model.pb` file is missing. Let's inspect the contents of the `poisoned_model` directory to find the correct path.

In [ ]:
print(f"Contents of {poisoned_model_path}:")
!ls -F "{poisoned_model_path}"

# If there are subdirectories (e.g., '1'), we might need to look inside them.
# For example:
# !ls -F "{poisoned_model_path}/1/"

### Image Preprocessing with OpenCV

Now, let's set up the code for image loading, cleaning, and basic preprocessing using OpenCV. The task mentions cleaning unwanted data like non-light/emitting pixels, offset, and grayscale values. Since the images are 16-bit grayscale PNGs, we'll focus on loading them correctly, normalizing pixel values, and demonstrating a simple cleaning step like thresholding or background removal.

We'll define the relevant paths and then create a function to process and display images.

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# Define the root folder path for all files
# Changed for Kaggle environment
root_folder_path = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'

# Corrected specific image set paths based on directory inspection
# For unlearn_set: images are directly in the unlearn_set folder
unlearn_set_path = os.path.join(root_folder_path, 'unlearn_set')

# For test_set: it seems to be nested within another 'test_set' folder.
# The path should be '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set'
test_set_path = os.path.join(root_folder_path, 'test_set', 'test_set')

print(f"Root Folder: {root_folder_path}")
print(f"Corrected Test Set Path: {test_set_path}")
print(f"Corrected Unlearn Set Path: {unlearn_set_path}")

# Reference path (as mentioned by the user, though its direct use for this task is unclear)
# sample_data_path = '/content/sample_data' # This path is for Colab, not Kaggle input. Can remove or comment out.


def process_and_display_image(image_path, title="Processed Image"):
    """
    Loads a 16-bit grayscale image, performs basic cleaning (normalization),
    and displays it.
    """
    if not os.path.exists(image_path):
        print(f"Error: Image not found at {image_path}")
        return None # Return None if image not found

    # Load the 16-bit grayscale image
    # cv2.IMREAD_UNCHANGED ensures it loads as 16-bit
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

    if image is None:
        print(f"Error: Could not load image from {image_path}")
        return None # Return None if image not loaded

    print(f"Original image shape: {image.shape}, dtype: {image.dtype}")

    # --- Image Cleaning / Preprocessing Steps ---
    # 1. Normalization: Scale 16-bit image (0-65535) to 0-1 for consistent processing
    #    and then to 0-255 for display if needed.
    normalized_image = image.astype(np.float32) / 65535.0

    # 2. Simple Thresholding (example for removing very dark/non-light pixels)
    #    This can help isolate brighter streak-like features from a dark background.
    #    Adjust threshold value (e.g., 0.01) based on image characteristics.
    threshold_value = 0.005 # Example: pixels below this value are set to 0 (black)
    cleaned_image = np.where(normalized_image > threshold_value, normalized_image, 0)

    # 3. Convert to 8-bit for display purposes (Matplotlib expects 0-255 or 0-1 float)
    display_image = (cleaned_image * 255).astype(np.uint8)

    # --- Display the image ---
    plt.figure(figsize=(8, 8))
    plt.imshow(display_image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

    print(f"Processed image shape: {cleaned_image.shape}, dtype: {cleaned_image.dtype}")
    return cleaned_image # Return the processed image if further steps are needed

### Correcting Image Paths

Based on the previous `ls` output, the image paths need adjustment. `unlearn_set` seems to contain images directly, while `test_set` has a nested `test_set` directory.

In [ ]:
import os

# Define the root folder path for all files (already defined, but good for context)
root_folder_path = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'

# Correct specific image set paths based on current directory inspection
# For unlearn_set: images are directly in the unlearn_set folder
unlearn_set_path = os.path.join(root_folder_path, 'unlearn_set')

# For test_set: it seems to be nested within another 'test_set' folder.
# We will try this path first based on the 'ls' output: 'test_set/test_set/'
test_set_path = os.path.join(root_folder_path, 'test_set', 'test_set')

print(f"Corrected Test Set Path: {test_set_path}")
print(f"Corrected Unlearn Set Path: {unlearn_set_path}")

# Re-run the image listing and processing for test_set with the corrected path
test_images = [f for f in os.listdir(test_set_path) if f.endswith('.png')]

if test_images:
    first_test_image = os.path.join(test_set_path, test_images[0])
    print(f"Processing example image: {first_test_image}")
    processed_img = process_and_display_image(first_test_image, title="Example Processed Test Image")
else:
    print(f"No PNG images found in {test_set_path} even after correction. Listing contents to debug...")
    !ls -F "{test_set_path}"


# Re-run the image listing and processing for unlearn_set with the corrected path
unlearn_images = [f for f in os.listdir(unlearn_set_path) if f.endswith('.png')]

if unlearn_images:
    first_unlearn_image = os.path.join(unlearn_set_path, unlearn_images[0])
    print(f"Processing example unlearn image: {first_unlearn_image}")
    processed_unlearn_img = process_and_display_image(first_unlearn_image, title="Example Processed Unlearn Image")
else:
    print(f"No PNG images found in {unlearn_set_path} even after correction. Listing contents to debug...")
    !ls -F "{unlearn_set_path}"


### Example Usage of Image Preprocessing

Let's pick an image from the `test_set` or `unlearn_set` and apply our `process_and_display_image` function. We first need to list the files in these directories to get an image name.

### Generating Grayscale, Black-and-White, and Inverted Images

To prepare the images for further processing as you described, we will generate three distinct representations for each: a normalized grayscale version, a binary (black background) version, and an inverted binary (white background) version. We'll use a simple threshold to create the binary images.

In [ ]:
### Image Representation and Feature Extraction Functions
# The `get_image_representations`, `display_image_set`, `get_2d_positional_encoding`, and `extract_streak_pixel_features` functions have been consolidated into a single utility functions cell to ensure consistent execution.

### Demonstration of Image Representations

Now, let's use these functions to process and display an example image from both the `test_set` and `unlearn_set`.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def get_image_representations(image_path):
    """
    Reads a 16-bit image and returns three representations:
    1. Normalized grayscale (0.0 - 1.0)
    2. Binary mask (Black: Threshold for dark features)
    3. Binary mask (White: Threshold for bright features)
    """
    # Load 16-bit image
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Could not load image at {image_path}")
        
    # 1. Normalize to 0-1 float
    norm_gray = img.astype(np.float32) / 65535.0
    
    # 2. Binary thresholding (adjust thresholds as needed for your debris)
    # Bright threshold (e.g., > 10% brightness)
    _, bin_white = cv2.threshold(img, int(65535 * 0.1), 255, cv2.THRESH_BINARY)
    # Dark threshold (e.g., < 2% brightness)
    _, bin_black = cv2.threshold(img, int(65535 * 0.02), 255, cv2.THRESH_BINARY_INV)
    
    return norm_gray, bin_black, bin_white

def display_image_set(path, norm, bin_b, bin_w, title):
    """Visualizes the representations."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(cv2.imread(path, cv2.IMREAD_UNCHANGED), cmap='gray')
    axes[0].set_title("Original")
    axes[1].imshow(norm, cmap='gray')
    axes[1].set_title("Normalized Gray")
    axes[2].imshow(bin_b, cmap='gray')
    axes[2].set_title("Dark Features Mask")
    axes[3].imshow(bin_w, cmap='gray')
    axes[3].set_title("Bright Features Mask")
    plt.suptitle(title)
    plt.show()

In [ ]:
# Get a list of image files from the test set
test_images = [f for f in os.listdir(test_set_path) if f.endswith('.png')]

if test_images:
    first_test_image_path = os.path.join(test_set_path, test_images[0])
    print(f"Processing example image from test_set: {first_test_image_path}")
    norm_gray_test, bin_black_test, bin_white_test = get_image_representations(first_test_image_path)
    display_image_set(first_test_image_path, norm_gray_test, bin_black_test, bin_white_test, "Test Set Image")
else:
    print(f"No PNG images found in {test_set_path}. Please check the path.")


# Get a list of image files from the unlearn set
unlearn_images = [f for f in os.listdir(unlearn_set_path) if f.endswith('.png')]

if unlearn_images:
    first_unlearn_image_path = os.path.join(unlearn_set_path, unlearn_images[0])
    print(f"Processing example image from unlearn_set: {first_unlearn_image_path}")
    norm_gray_unlearn, bin_black_unlearn, bin_white_unlearn = get_image_representations(first_unlearn_image_path)
    display_image_set(first_unlearn_image_path, norm_gray_unlearn, bin_black_unlearn, bin_white_unlearn, "Unlearn Set Image")
else:
    print(f"No PNG images found in {unlearn_set_path}. Please check the path.")

In [ ]:
# Get a list of image files from the test set
test_images = [f for f in os.listdir(test_set_path) if f.endswith('.png')]

if test_images:
    first_test_image = os.path.join(test_set_path, test_images[0])
    print(f"Processing example image: {first_test_image}")
    processed_img = process_and_display_image(first_test_image, title="Example Processed Test Image")
else:
    print(f"No PNG images found directly in {test_set_path}. Listing contents to debug...")
    !ls -F "{test_set_path}"

# You can similarly process an image from unlearn_set if needed
# unlearn_images = [f for f in os.listdir(unlearn_set_path) if f.endswith('.png')]
# if unlearn_images:
#     first_unlearn_image = os.path.join(unlearn_set_path, unlearn_images[0])
#     print(f"Processing example unlearn image: {first_unlearn_image}")
#     processed_unlearn_img = process_and_display_image(first_unlearn_image, title="Example Processed Unlearn Image")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureEmbedding(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim)
        )
    def forward(self, x):
        return self.proj(x)

class GraphAttentionLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

class GraphEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GraphAttentionLayer(embed_dim, num_heads, dropout_rate) for _ in range(num_layers)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class DetectionHead(nn.Module):
    def __init__(self, embed_dim, num_queries, dropout_rate=0.1):
        super().__init__()
        self.query_embed = nn.Parameter(torch.randn(1, num_queries, embed_dim))
        self.bbox_pred = nn.Sequential(nn.Linear(embed_dim, 4), nn.Sigmoid())
        self.class_pred = nn.Linear(embed_dim, 1)
    def forward(self, encoded_features):
        batch_size = encoded_features.size(0)
        queries = self.query_embed.expand(batch_size, -1, -1)
        # Simplified query-based detection for demonstration
        boxes = self.bbox_pred(queries)
        confs = torch.sigmoid(self.class_pred(queries))
        return boxes, confs

print("Model classes defined successfully.")

In [ ]:
import torch
import numpy as np
import cv2

def extract_streak_pixel_features(norm_gray, bin_black, bin_white, pos_enc_dim=64):
    """
    Identifies streak pixels and creates a feature matrix.
    """
    # 1. Identify indices where pixels are part of a streak
    mask = (bin_black > 0) | (bin_white > 0)
    y_coords, x_coords = np.where(mask)
    
    # 2. Build features: [intensity, binary_b, binary_w, y_norm, x_norm]
    num_pixels = len(y_coords)
    features = np.zeros((num_pixels, 5), dtype=np.float32)
    
    features[:, 0] = norm_gray[mask]
    features[:, 1] = bin_black[mask] / 255.0
    features[:, 2] = bin_white[mask] / 255.0
    features[:, 3] = y_coords / norm_gray.shape[0]
    features[:, 4] = x_coords / norm_gray.shape[1]
    
    return torch.tensor(features), torch.tensor(np.stack([y_coords, x_coords], axis=1))

print("Feature extraction function defined.")

In [ ]:
# Assuming norm_gray_test, bin_black_test, bin_white_test are already loaded
if 'norm_gray_test' in locals():
    temp_node_features, temp_pixel_coords = extract_streak_pixel_features(
        norm_gray_test, bin_black_test, bin_white_test, pos_enc_dim=64
    )
    
    # Sample to avoid OOM
    MAX_DEMO_PIXELS = 2000
    if temp_node_features.shape[0] > MAX_DEMO_PIXELS:
        indices = torch.randperm(temp_node_features.shape[0])[:MAX_DEMO_PIXELS]
        streak_node_features = temp_node_features[indices]
    else:
        streak_node_features = temp_node_features
        
    print(f"Features generated: {streak_node_features.shape}")

In [ ]:
# --- Execution & Demonstration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'streak_node_features' not in locals():
    print("Error: 'streak_node_features' not found. Please run your feature extraction cell first.")
else:
    # Hyperparameters
    embed_dim = 128
    num_heads = 8
    num_layers = 3
    num_queries = 10
    
    # Initialize components
    embedder = FeatureEmbedding(streak_node_features.shape[1], embed_dim).to(device)
    encoder = GraphEncoder(embed_dim, num_heads, num_layers).to(device)
    head = DetectionHead(embed_dim, num_queries).to(device)
    
    # Pipeline
    with torch.no_grad():
        features = embedder(streak_node_features.to(device).unsqueeze(0))
        encoded = encoder(features)
        boxes, confs = head(encoded)
        
    print(f"Pipeline Complete.")
    print(f"Boxes shape: {boxes.shape} | Confidences shape: {confs.shape}")
    print("\nSample Bounding Boxes (cx, cy, w, h):")
    print(boxes[0][:5].cpu().numpy())

In [ ]:
# Ensure previous steps ran
if 'streak_node_features' in locals():
    # 1. Embedding
    embedder = FeatureEmbedding(5, 128).to(device)
    embedded_features = embedder(streak_node_features.to(device).unsqueeze(0))
    
    # 2. Encoder
    encoder = GraphEncoder(128, num_heads=8, num_layers=3).to(device)
    final_encoded = encoder(embedded_features)
    
    # 3. Head
    head = DetectionHead(128, num_queries=10).to(device)
    boxes, confs = head(final_encoded)
    
    print("Pipeline finished successfully.")
else:
    print("Please run Cell 1 and Cell 2 first!")

### Inspecting Image Directories

It seems no PNG images were directly found in the `test_set` and `unlearn_set` paths. This might mean the images are located in subdirectories within these folders. Let's list the contents to find the actual image locations.

In [ ]:
print(f"Contents of {test_set_path}:")
!ls -F "{test_set_path}"

print(f"\nContents of {unlearn_set_path}:")
!ls -F "{unlearn_set_path}"

print(f"\nContents of {root_folder_path}:")
!ls -F "{root_folder_path}"


In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/sample_submission.csv") # Updated path
sub.head()

In [ ]:
sub.tail()

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm

In [ ]:
# ==============================================================================
# 1. KAGGLE PATH CONFIGURATIONS
# ==============================================================================
# Define the root paths strictly for the Kaggle environment
BASE_DIR = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
TEST_SET_PATH = os.path.join(BASE_DIR, "test_set", "test_set")
UNLEARN_SET_PATH = os.path.join(BASE_DIR, "unlearn_set")
MODEL_PATH = os.path.join(BASE_DIR, "poisoned_model", "poisoned_model.pth")
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Hyperparameters for T4x2 GPU Limits
MAX_PIXELS = 2000      # Prevent OOM on Attention mechanisms
INPUT_DIM = 67         # Matching the node feature dimension from your notebook
EMBED_DIM = 128
NUM_HEADS = 8
NUM_LAYERS = 3
NUM_QUERIES = 10       # Max detections per image
CONF_THRESHOLD = 0.5   # Threshold to include a box in the submission

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ==============================================================================
# 2. FULL DEEP LEARNING MODEL ARCHITECTURE
# ==============================================================================
class FeatureEmbedding(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, x):
        return self.proj(x)

class GraphAttentionLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # Self-attention over the nodes (pixels)
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

class GraphEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GraphAttentionLayer(embed_dim, num_heads, dropout_rate) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class DetectionHead(nn.Module):
    def __init__(self, embed_dim, num_queries, num_bbox_params=4, dropout_rate=0.1):
        super().__init__()
        # Learnable object queries to find streaks
        self.query_embed = nn.Parameter(torch.randn(1, num_queries, embed_dim))
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads=8, dropout=dropout_rate, batch_first=True)
        
        # Bounding box regressor (outputs normalized cx, cy, w, h)
        self.bbox_pred = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, num_bbox_params),
            nn.Sigmoid() 
        )
        # Class/Confidence regressor
        self.class_pred = nn.Linear(embed_dim, 1)

    def forward(self, encoded_features):
        batch_size = encoded_features.size(0)
        queries = self.query_embed.expand(batch_size, -1, -1)
        
        # Cross attention: queries attend to the encoded graph features
        attn_out, _ = self.cross_attn(queries, encoded_features, encoded_features)
        
        boxes = self.bbox_pred(attn_out)
        confidences = torch.sigmoid(self.class_pred(attn_out))
        return boxes, confidences

class GraphDeepLearningModel(nn.Module):
    def __init__(self, input_feature_dim, embed_dim, num_graph_layers, num_heads, num_queries, num_bbox_params=4, dropout_rate=0.1):
        super().__init__()
        self.embedding = FeatureEmbedding(input_feature_dim, embed_dim)
        self.encoder = GraphEncoder(embed_dim, num_heads, num_graph_layers, dropout_rate)
        self.head = DetectionHead(embed_dim, num_queries, num_bbox_params, dropout_rate)

    def forward(self, x):
        # If input lacks batch dimension, add it
        if x.dim() == 2:
            x = x.unsqueeze(0)
            
        emb = self.embedding(x)
        enc = self.encoder(emb)
        boxes, confs = self.head(enc)
        return boxes, confs

In [ ]:
# ==============================================================================
# 3. FEATURE EXTRACTION PIPELINE
# ==============================================================================
def process_and_extract_features(image_path, max_pixels=MAX_PIXELS, input_dim=INPUT_DIM):
    """
    Loads a 16-bit PNG, identifies streak-like pixels via thresholding,
    and returns simulated 67-dimensional node features mapped to coordinates.
    """
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None, None
        
    img_h, img_w = img.shape
    normalized_image = img.astype(np.float32) / 65535.0
    
    # Isolate brighter features
    mask = normalized_image > 0.005
    y_coords, x_coords = np.where(mask)
    intensities = normalized_image[mask]
    
    if len(intensities) == 0:
        return None, (img_w, img_h)
        
    # Subsample to avoid CUDA OOM errors
    if len(intensities) > max_pixels:
        indices = np.random.choice(len(intensities), max_pixels, replace=False)
        y_coords = y_coords[indices]
        x_coords = x_coords[indices]
        intensities = intensities[indices]
        
    # Build feature matrix (Padding to 67 dims to fit original notebook structure)
    node_features = np.zeros((len(intensities), input_dim), dtype=np.float32)
    node_features[:, 0] = intensities
    node_features[:, 1] = y_coords / img_h  # Normalized Y
    node_features[:, 2] = x_coords / img_w  # Normalized X
    
    return torch.tensor(node_features, dtype=torch.float32), (img_w, img_h)

In [ ]:
# ==============================================================================
# 4. INFERENCE AND SUBMISSION CREATION (CLEANED)
# ==============================================================================
def generate_submission():
    print("Initializing Model...")
    model = GraphDeepLearningModel(
        input_feature_dim=INPUT_DIM,
        embed_dim=EMBED_DIM,
        num_graph_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        num_queries=NUM_QUERIES
    )
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    model.to(device)
    
    # Load weights
    if os.path.exists(MODEL_PATH):
        try:
            state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)
            model.load_state_dict(state_dict, strict=False)
            print("Model weights loaded.")
        except Exception as e:
            print(f"Load warning: {e}")
            
    model.eval()
    results = []
    
    print("Running Inference on Test Set...")
    for img_id in tqdm(range(2000)):
        img_name = f"{img_id}.png"
        img_path = os.path.join(TEST_SET_PATH, img_name)
        
        # Default to a single space (Kaggle requirement for null/empty)
        prediction_string = " " 
        
        if os.path.exists(img_path):
            features, dims = process_and_extract_features(img_path)
            
            if features is not None and features.shape[0] > 0:
                img_w, img_h = dims
                features = features.to(device)
                
                with torch.no_grad():
                    boxes, confs = model(features) 
                    
                # Extract and flatten
                boxes = boxes.cpu().numpy()[0]
                confs = confs.cpu().numpy().flatten()
                
                detections = []
                for box, conf in zip(boxes, confs):
                    if float(conf) > CONF_THRESHOLD:
                        cx, cy, nw, nh = box # Unpack once
                        
                        w_abs = int(nw * img_w)
                        h_abs = int(nh * img_h)
                        x_top = int(max(0, (cx * img_w) - (w_abs / 2.0)))
                        y_top = int(max(0, (cy * img_h) - (h_abs / 2.0)))
                        
                        detections.append(f"{float(conf):.4f} {x_top} {y_top} {w_abs} {h_abs}")
                
                if detections:
                    prediction_string = " ".join(detections)
                    
        results.append({"image_id": img_id, "prediction_string": prediction_string})
        
    # Create DataFrame explicitly to ensure column order
    submission_df = pd.DataFrame(results, columns=['image_id', 'prediction_string'])
    submission_df = submission_df.rename(columns={'image_id': 'id'})
    # Save to CSV
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    
    print(f"\nInference Complete! Saved to: {SUBMISSION_PATH}")
    print("CSV Header check:", pd.read_csv(SUBMISSION_PATH).columns.tolist())
    # Display the first few rows of the actual saved CSV to verify
    print("=========================================")
    print("|",pd.read_csv(SUBMISSION_PATH).head(),"|")
    print("=========================================")
    print("|",pd.read_csv(SUBMISSION_PATH).tail(),"|")
    print("=========================================")

if __name__ == "__main__":
    generate_submission()